In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by using a `while True` loop and checking whether the response contained any `function_call` items.

- It sends the current `messages` to the model.
- It appends the model’s output to `messages`.
- If there’s a `function_call`, it runs the tool, appends the tool result, and sets `has_function_calls = True`.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps looping until the model returns a response with no more tool calls.


In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by using a `while True` loop and a `has_function_calls` flag.

- It sends the current `messages` to the model.
- If the model returns any `function_call`, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- If there are no function calls in that response, the code `break`s out of the loop.

So the stop condition is: **no function calls this turn**.


# Question 1

3

# Question 2

7000

# Question 3 

Over 2000ms

# Question 4


In [4]:
import sqlite3

conn = sqlite3.connect("traces.db")

cursor = conn.execute(
    "SELECT name FROM spans"
)

cursor.fetchall()

[('search',), ('llm',), ('rag',)]

# Question 5

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

df = pd.read_sql(
    "SELECT * FROM spans",
    conn
)

df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784496680759470379,1784496680761390742,NaN,NaN,None
1,llm,1784496680773535114,1784496682909325478,7111.0,93.0,None
2,rag,1784496680759390601,1784496682923899335,NaN,NaN,None
3,search,1784496853655688442,1784496853657501075,NaN,NaN,None
4,llm,1784496853670282776,1784496856071329257,7111.0,122.0,None
5,rag,1784496853655607564,1784496856088446435,NaN,NaN,None


In [6]:
df["duration_ms"] = (
    df["end_time"] - df["start_time"]
) / 1_000_000

In [7]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost,duration_ms
0,search,1784496680759470379,1784496680761390742,NaN,NaN,None,1.920363
1,llm,1784496680773535114,1784496682909325478,7111.0,93.0,None,2135.790364
2,rag,1784496680759390601,1784496682923899335,NaN,NaN,None,2164.508734
3,search,1784496853655688442,1784496853657501075,NaN,NaN,None,1.812633
4,llm,1784496853670282776,1784496856071329257,7111.0,122.0,None,2401.046481
5,rag,1784496853655607564,1784496856088446435,NaN,NaN,None,2432.838871


In [8]:
df[df.name != "rag"].groupby("name")["duration_ms"].sum()

name
llm       4536.836845
search       3.732996
Name: duration_ms, dtype: float64

It's LLM

# Question 6

In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

df = pd.read_sql(
    "SELECT * FROM spans",
    conn
)

df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784496680759470379,1784496680761390742,NaN,NaN,None
1,llm,1784496680773535114,1784496682909325478,7111.0,93.0,None
2,rag,1784496680759390601,1784496682923899335,NaN,NaN,None
3,search,1784496853655688442,1784496853657501075,NaN,NaN,None
4,llm,1784496853670282776,1784496856071329257,7111.0,122.0,None
5,rag,1784496853655607564,1784496856088446435,NaN,NaN,None
6,search,1784497130905287204,1784497130906905104,NaN,NaN,None
7,llm,1784497130920320030,1784497133171363385,7111.0,132.0,None
8,rag,1784497130905207674,1784497133183679136,NaN,NaN,None


In [10]:
llm_df = df[df["name"] == "llm"]

llm_df["input_tokens"]

1    7111.0
4    7111.0
7    7111.0
Name: input_tokens, dtype: float64

They're identical